In [5]:
from clickhouse_driver import Client
import pandas as pd
user = input('Введите имя пользователя')
password = input('Введите пароль')
database = input('Введите назв-е БД')


client = Client(host='clickhouse.lab.karpov.courses', port=9000, user=user,
                password=password, database=database)

def get_data(query):
    """
    Вытягивает данные из clickhouse в виде Dataframe
    
    query - запрос
    """
    result, columns = client.execute(query, with_column_types=True)
    return pd.DataFrame(result, columns=[tuple[0] for tuple in columns])


In [6]:
client.execute('''CREATE TABLE hardda_student_data.user_dm_events_al_ivanenko
(
    event_date Date ,
    week_start_date Date ALIAS toMonday(event_date),
    
    platform LowCardinality(String),
    
    user_pseudo_id UInt256  CODEC(ZSTD),
    user_x_phone_id UInt256  CODEC(ZSTD),
    
    cnt_events UInt16, 
    cnt_view_advertisement UInt16, 
    cnt_view_listing UInt16, 
    cnt_new_advertisement_open UInt16, 
    cnt_new_advertisement_view_screen UInt16, 
    cnt_successful_new_advertisement_creation UInt16,
    cnt_session_initiation UInt8, 
    cnt_display_phone UInt16, 
    cnt_send_message UInt16, 
    cnt_order_via_phone UInt8,
    cnt_add_to_favorites UInt16, 
    cnt_view_ads_in_cabinet UInt16,
    cnt_edit_advert_view_screen_package UInt16,
    cnt_new_advert_view_screen_package UInt16
    
    
)
ENGINE = MergeTree()
ORDER BY (platform, user_pseudo_id, user_x_phone_id, event_date)
SETTINGS 
    ratio_of_defaults_for_sparse_serialization = 0.89,
    index_granularity = 8192
AS
    SELECT event_date, platform
        , reinterpretAsUInt256(reverse(unhex(user_pseudo_id)))
        , reinterpretAsUInt256(reverse(unhex(replaceAll(user_x_phone_id, '-','') )))
        , cnt_events, cnt_view_advertisement, cnt_view_listing
        , cnt_new_advertisement_open, cnt_new_advertisement_view_screen
        , cnt_successful_new_advertisement_creation
        , cnt_session_initiation, cnt_display_phone
        , cnt_send_message, cnt_order_via_phone
        , cnt_add_to_favorites, cnt_view_ads_in_cabinet
        , cnt_edit_advert_view_screen_package
        , cnt_new_advert_view_screen_package
    from hardda.user_dm_events
               
              ''')



ServerException: Code: 57.
DB::Exception: Table hardda_student_data.user_dm_events_al_ivanenko already exists. Stack trace:

0. DB::Exception::Exception(DB::Exception::MessageMasked&&, int, bool) @ 0x000000000fbffd1b
1. DB::Exception::Exception(PreformattedMessage&&, int) @ 0x0000000009d9e38c
2. DB::Exception::Exception<String, String>(int, FormatStringHelperImpl<std::type_identity<String>::type, std::type_identity<String>::type>, String&&, String&&) @ 0x0000000009d9de6b
3. DB::InterpreterCreateQuery::doCreateTable(DB::ASTCreateQuery&, DB::InterpreterCreateQuery::TableProperties const&, std::unique_ptr<DB::DDLGuard, std::default_delete<DB::DDLGuard>>&, DB::LoadingStrictnessLevel) @ 0x0000000013dbe6bb
4. DB::InterpreterCreateQuery::createTable(DB::ASTCreateQuery&) @ 0x0000000013db0550
5. DB::InterpreterCreateQuery::execute() @ 0x0000000013dc4298
6. DB::executeQueryImpl(char const*, char const*, std::shared_ptr<DB::Context>, DB::QueryFlags, DB::QueryProcessingStage::Enum, DB::ReadBuffer*

In [7]:
get_data('''
    SELECT 
         countIf(cnt_order_via_phone = 0) * 1.0 / count() as correct_dolya1 
        , countIf(cnt_add_to_favorites = 0) * 1.0 / count() as correct_dolya2
        , countIf(cnt_new_advertisement_view_screen = 0) * 1.0 / count() as correct_dolya3
        , countIf(cnt_display_phone = 0) * 1.0 / count() as correct_dolya4
        , countIf(cnt_view_ads_in_cabinet = 0) * 1.0 / count() as correct_dolya5
        , countIf(cnt_send_message = 0) * 1.0 / count() as correct_dolya6
        , countIf(cnt_new_advert_view_screen_package = 0) * 1.0 / count() as correct_dolya7
        , countIf(cnt_successful_new_advertisement_creation = 0) * 1.0 / count() as correct_dolya8
        , countIf(cnt_events = 0) * 1.0 / count() as correct_dolya9
        , countIf(cnt_view_advertisement = 0) * 1.0 / count() as correct_dolya10
        , countIf(cnt_view_listing = 0) * 1.0 / count() as correct_dolya11
        , countIf(cnt_session_initiation = 0) * 1.0 / count() as correct_dolya12
        , countIf(cnt_edit_advert_view_screen_package = 0) * 1.0 / count() as correct_dolya13
        , countIf(cnt_new_advertisement_open = 0) * 1.0 / count() as correct_dolya14
        
        
    FROM hardda_student_data.user_dm_events_al_ivanenko
''')

,correct_dolya1,correct_dolya2,correct_dolya3,correct_dolya4,correct_dolya5,correct_dolya6,correct_dolya7,correct_dolya8,correct_dolya9,correct_dolya10,correct_dolya11,correct_dolya12,correct_dolya13,correct_dolya14
0,0.999876,0.876092,0.977869,0.891849,0.892679,0.90937,0.978034,0.977981,0.0,0.233271,0.47144,0.030247,0.984847,0.984738


In [8]:
get_data('''
    SELECT name, formatReadableSize(a1.data_compressed_bytes) as my_size
        , formatReadableSize(a2.data_compressed_bytes) as karpov_size
    from system.columns a1
    LEFT JOIN system.columns as a2
        ON a1.name = a2.name
        AND a2.database = 'hardda'
        AND a2.table = 'user_dm_events'
    WHERE database = 'hardda_student_data' 
        AND table = 'user_dm_events_al_ivanenko'

''')

,name,my_size,karpov_size
0,event_date,80.50 MiB,437.58 KiB
1,week_start_date,0.00 B,437.55 KiB
2,platform,256.81 KiB,51.38 MiB
3,user_pseudo_id,276.24 MiB,1.70 GiB
4,user_x_phone_id,327.14 MiB,1.82 GiB
5,cnt_events,82.53 MiB,134.06 MiB
6,cnt_view_advertisement,76.02 MiB,124.09 MiB
7,cnt_view_listing,63.81 MiB,94.13 MiB
8,cnt_new_advertisement_open,1.52 MiB,4.60 MiB
9,cnt_new_advertisement_view_screen,2.64 MiB,6.40 MiB


In [9]:
get_data('''
    SELECT name , formatReadableSize(total_bytes) FROM system.tables
    WHERE database = 'hardda_student_data'
        AND name = 'user_dm_events_al_ivanenko'
        
    UNION ALL 
    
    SELECT name, formatReadableSize(total_bytes) FROM system.tables
    WHERE database = 'hardda'
    
        AND name = 'user_dm_events' 
    '''
        )

,name,formatReadableSize(total_bytes)
0,user_dm_events,4.72 GiB
1,user_dm_events_al_ivanenko,1006.59 MiB


In [ ]:

 git branch -M main 
git push -u origin main

SyntaxError: invalid syntax (3067870759.py, line 1)